# Extract and Regrid ERA5 data
Code to extract era5 data for a specified sub-region and regrid to a coarser grid and also coarser time step

This code is used again to apply to QTRACK software. Currently set to extract 6 hrly era5 data. Software suggests one week of reanalysis data to merge with model data to give tracker enough time to spin up. Make sure to adjust dates accordingly.

You will need to run the first part of this code twice: once for u and once for v files.

- NCSU Large Scale and Tropical Dynamics

Versions
- A. Aiyyer, Jul 23, 2023
- A. Thornton, Sep 14, 2023


In [4]:
import numpy as np
import xarray as xr
import pandas as pd
from datetime import date
from numpy import absolute, exp, log

# Any import of metpy will activate the accessors
from metpy.units import units
import os
import glob

# for regridding
import xesmf as xe

### Paths to find and save data

In [5]:
# daily era5
era5_sfc_dir = '/glade/campaign/collections/rda/data/d633000/e5.oper.an.sfc/'
era5_pl_dir  = '/glade/campaign/collections/rda/data/d633000/e5.oper.an.pl/'


# output path to save regridded data
path_out = '/glade/u/home/athornton/qtrack/wind_files/u_v_era5/'

### Select variable
Pick one of the following 8 variables, uncomment the three lines representing the variable.

In [10]:
#varId  = '129'
#varNam = 'z'
#variab = 'Z'  # the variable name in the data file

#varId  = '130'
#varNam = 't'
#variab = 'T'

#varId  = '060'
#varNam = 'pv'
#variab = 'PV'

#varId  = '138'
#varNam = 'vo'
#variab = 'VO'

# varId  = '133'
# varNam = 'q'
# variab = 'Q'

#varId  = '135'
#varNam = 'w'
#variab = 'W'

varId  = '131'
varNam = 'u'
variab = 'U'

# varId  = '132'
# varNam = 'v'
# variab = 'V'

### Select subset of data
Define the specifications for subset of data: Pick latitude and longitude bounds, new grid spacing, lower and upper bound of levels, and range of dates.

In [11]:
# lat/lons
latS = -1.
latN =  36.
lonW = -120.
lonE =  19.

# grid spacing
dlat = 1
dlon = 1

# levels
level = 700

# range of dates
year_start  = 2020
month_start = 8
day_start   = 24

year_end  = 2020
month_end = 9
day_end   = 9

date_series = [pd.date_range(date(i,month_start,day_start),date(i,month_end,day_end), freq ='D') for i in range(year_start,year_end+1)]
# date_series is a list of lists. Lets unpack it now
dates_list = [element for sublist in date_series for element in sublist]

print(dates_list[0].strftime("%Y%m%d"))
print(dates_list[-1].strftime("%Y%m%d"))


20200824
20200909


### Function for regridding and applying specifications for subset of data, and downloading and writing new netcdf files.

In [12]:
firstPass = True

for a_date in dates_list:
    #print( a_date.strftime('%Y%m%d') )
    b_date = a_date + pd.DateOffset(hours=23)
    times = [a_date + pd.DateOffset(hour=h) for h in np.arange(0,24,6)]

    fname =   a_date.strftime('%Y%m') + '/e5.oper.an.pl.128_'+varId+'_'+varNam+'.ll025uv.' \
    +   a_date.strftime('%Y%m%d')  + '00_' +  a_date.strftime('%Y%m%d') + '23.nc'
    
    infile = era5_pl_dir + fname    

    
    ds  = xr.open_dataset(infile)  
    
    # prepare to roll the longitude from 0 to 360 --> -180 to 180
    ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
    
    dat = ds[variab].sel(level=level, time=times, latitude=slice(latN,latS))
    dat = dat.roll(longitude=int(len(dat['longitude']) / 2), roll_coords=True)
    
    
    # create regriddger only once and reuse it afterward
    if (firstPass):
        ds_out = xr.Dataset( 
            {
                "latitude": (["latitude"], np.arange(latN,latS, -dlat),  {"units": "degrees_north"}),
                "longitude": (["longitude"], np.arange(lonW, lonE, dlon), {"units": "degrees_east"}),

            }
        )
        ds_in = xr.Dataset(
            {
                "latitude": (["latitude"], dat.latitude.values,  {"units": "degrees_north"}),
                "longitude": (["longitude"], dat.longitude.values, {"units": "degrees_east"}),
            }
        )
        regridder = xe.Regridder(ds_in, ds_out, "conservative")
        firstPass = False
        
        
    # regrid and write to file (4x daily)
    dat_out = regridder(dat, keep_attrs=True)    
    file_out = path_out + varNam+ '_obs_' + a_date.strftime("%Y%m%d") + '.nc'
    dat_out.to_netcdf(path=file_out)
print('success, '+varNam+' files made')


success, u files made


In [1]:
# file_strings = []
# for i in range(0,len(dates_list)):
#     string = dates_list[i].strftime("%Y%m%d")
#     file_strings.append(string)

In [2]:
# data_files_u  = [ '/glade/u/home/athornton/qtrack/wind_files/u_v_era5/u_obs_' 
#                  + str(date) + '.nc' for date in file_strings ]

In [3]:
# data_files_v  = [ '/glade/u/home/athornton/qtrack/wind_files/u_v_era5/v_obs_' 
#                  + str(date) + '.nc' for date in file_strings ]

In [4]:
# u_wind = xr.open_mfdataset(data_files_u)
# v_wind = xr.open_mfdataset(data_files_v)


## BEFORE RUNNING THE CELLS BELOW:
Ensure that you have run the first part of this code for both u and for v. Once these files have been created, then you can grab them from the saved wind directory and create one 6hrly 'wind_season_lowres_700_2020_B1-6hr.nc' file to merge with the model data.  

In [13]:
v_wind = xr.open_mfdataset('/glade/u/home/athornton/qtrack/wind_files/u_v_era5/v_2020*')

In [14]:
u_wind = xr.open_mfdataset('/glade/u/home/athornton/qtrack/wind_files/u_v_era5/u_2020*')

In [15]:
ds = xr.Dataset( 
    data_vars=dict(
        u=(["time","latitude","longitude"], u_wind.U.compute().data),
        v=(["time","latitude","longitude"], v_wind.V.compute().data),
    ),
    coords=dict(
        longitude=("longitude", u_wind.longitude.values),
        latitude=("latitude", u_wind.latitude.values),
        time=("time", u_wind.time.values),
    ),
)

In [16]:
#file_out = path_out+'../wind_obs_highres_'+str(level)+'_3hr2.nc'
file_out = path_out+'../wind_season_lowres_700_2020_B1-6hr.nc'
ds.to_netcdf(path=file_out, format='NETCDF4', mode='w')

In [17]:
ds = xr.open_dataset(file_out)
ds

<xarray.Dataset> Size: 3MB
Dimensions:    (time: 68, latitude: 37, longitude: 139)
Coordinates:
  * longitude  (longitude) float64 1kB -120.0 -119.0 -118.0 ... 16.0 17.0 18.0
  * latitude   (latitude) float64 296B 36.0 35.0 34.0 33.0 ... 3.0 2.0 1.0 0.0
  * time       (time) datetime64[ns] 544B 2020-08-24 ... 2020-09-09T18:00:00
Data variables:
    u          (time, latitude, longitude) float32 1MB ...
    v          (time, latitude, longitude) float32 1MB ...